In [14]:
import os
import pandas as pd


amz = pd.read_parquet('amazon.parquet')
meta = pd.read_parquet('metabooks.parquet')
good = pd.read_parquet('goodreads.parquet')

amz.shape, meta.shape, good.shape

((271360, 6), (535159, 12), (52478, 14))

In [15]:
data_list = []
for f in os.listdir('../ml-datasets/'):
  if f.endswith('.csv'):
    data_list.append(pd.read_csv(os.path.join('../ml-datasets/', f)))
data = pd.concat(data_list, axis=0)

In [16]:
id_list = list(set(data.id_left.tolist() + data.id_right.tolist()))

In [17]:
isbn_m2a = set(meta['isbn_clean']) & set(amz['isbn_clean'])
isbn_m2g = set(meta['isbn_clean']) & set(good['isbn_clean'])


meta_isbn = meta[meta['isbn_clean'].isin(isbn_m2a)]
amz_isbn = amz[amz['isbn_clean'].isin(isbn_m2a)]
good_isbn = good[good['isbn_clean'].isin(isbn_m2g)]

meta_id = meta[meta['id'].isin(id_list)]
amz_id = amz[amz['id'].isin(id_list)]
good_id = good[good['id'].isin(id_list)]


meta_sample = pd.concat([meta_isbn, meta_id], axis=0).drop_duplicates()
amz_sample = pd.concat([amz_isbn, amz_id], axis=0).drop_duplicates()
good_sample = pd.concat([good_isbn, good_id], axis=0).drop_duplicates()

meta_sample.shape, amz_sample.shape, good_sample.shape

((28038, 12), (27745, 6), (6650, 14))

In [18]:
sample_size = 40_000

remaining_meta = meta.drop(meta_sample.index)
remaining_amz = amz.drop(amz_sample.index)
remaining_good = good.drop(good_sample.index)

datasets = {
    "metabooks": (meta_sample, remaining_meta),
    "amazon": (amz_sample, remaining_amz),
    "goodreads": (good_sample, remaining_good),
}


dfs = {}

for name, (sample, remaining) in datasets.items():
  df = remaining.sample(
    n=sample_size - len(sample),
    random_state=42
  )

  dfs[name] = pd.concat([sample, df], axis=0)


In [ ]:
os.makedirs('./sample_data', exist_ok=True)

for name, df in dfs.items():
  df.to_csv(os.path.join('sample_data', f'{name}_40k.csv'), index=False)